# Chronos-T5 Base (200M): M6 inference, rounds 1 to 12

Zero-shot inference for all 12 official M6 rounds. Each round reads its
pre-built context of 512 weekday log returns, draws 100 sampled 20-day
trajectories per asset, and saves the raw array unchanged for later scoring.

| | |
|---|---|
| Model | `amazon/chronos-t5-base`, zero-shot, no fine-tuning |
| Samples | 100 trajectories per asset |
| Fixed | context 512 weekdays, horizon 20, seed 42, official M6 asset order |
| Input | the 12 round-context CSVs on Drive |
| Output | 12 NPZ files plus one inference report |

Nothing is post-processed here. Quintile probabilities and RPS are computed
later by `scripts/evaluate_m6_rps.py`, so this notebook only has to produce
correct raw forecasts.

Each round is saved and verified before the next begins, so a Colab disconnect
costs only the round in progress.

## 1. Install Chronos

Colab may ask you to restart the runtime. If it does, restart and re-run from
this cell.

In [ ]:
%pip install -q chronos-forecasting

## 2. Imports and settings

The round schedule and expected missing-history pattern are fixed here, so a
wrong or re-saved context file is caught before any inference runs.

In [ ]:
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import ChronosPipeline

# Experiment settings, unchanged from the original round-by-round runs.
MODEL_ID = "amazon/chronos-t5-base"
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 20
NUM_SAMPLES = 100
RANDOM_SEED = 42

# Assets per forward pass. Lower this if Colab reports a CUDA OOM error.
# It changes speed/memory ONLY - never the results' shape or ordering.
SERIES_BATCH_SIZE = 10

ROUNDS = range(1, 13)

# round -> context start, origin (= context end), forecast start, forecast end
ROUND_SCHEDULE = {
    1:  ("2020-03-19", "2022-03-04", "2022-03-07", "2022-04-01"),
    2:  ("2020-04-16", "2022-04-01", "2022-04-04", "2022-04-29"),
    3:  ("2020-05-14", "2022-04-29", "2022-05-02", "2022-05-27"),
    4:  ("2020-06-11", "2022-05-27", "2022-05-30", "2022-06-24"),
    5:  ("2020-07-09", "2022-06-24", "2022-06-27", "2022-07-22"),
    6:  ("2020-08-06", "2022-07-22", "2022-07-25", "2022-08-19"),
    7:  ("2020-09-03", "2022-08-19", "2022-08-22", "2022-09-16"),
    8:  ("2020-10-01", "2022-09-16", "2022-09-19", "2022-10-14"),
    9:  ("2020-10-29", "2022-10-14", "2022-10-17", "2022-11-11"),
    10: ("2020-11-26", "2022-11-11", "2022-11-14", "2022-12-09"),
    11: ("2020-12-24", "2022-12-09", "2022-12-12", "2023-01-06"),
    12: ("2021-01-21", "2023-01-06", "2023-01-09", "2023-02-03"),
}

# Genuine leading missing history, per context-building step (verified locally).
# CARR is complete from round 2 onward; OGN (listed May 2021) gains 20 valid
# returns per round, so its leading NaN block shrinks by 20 each round.
EXPECTED_LEADING_NAN = {r: {"OGN": 302 - 20 * (r - 1)} for r in ROUNDS}
# CARR has one missing day in round 1 only.
EXPECTED_LEADING_NAN[1] = {"CARR": 1, "OGN": 302}

# Official M6 asset order, used to verify the contexts rather than reorder them.
OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

print(f"chronos-forecasting {version('chronos-forecasting')} | torch {torch.__version__}")
print(f"Experiment: {MODEL_ID} | rounds {min(ROUNDS)}-{max(ROUNDS)} ({len(list(ROUNDS))} rounds)")
print(f"Target raw output shape per round: ({N_ASSETS}, {NUM_SAMPLES}, {PREDICTION_LENGTH})")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configure paths

Contexts are read from the shared folder used by every notebook in this
project. The output folder is created automatically.

In [ ]:
DRIVE_CONTEXT_DIR = Path("/content/drive/MyDrive/HonoursResearch/contexts")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/HonoursResearch/outputs/chronos_t5_base/samples_100")


def context_path(round_number: int) -> Path:
    return DRIVE_CONTEXT_DIR / f"round_{round_number:02d}_context.csv"


def samples_path(round_number: int) -> Path:
    return DRIVE_OUTPUT_DIR / f"chronos_t5_base_round{round_number:02d}_samples.npz"


REPORT_PATH = DRIVE_OUTPUT_DIR / "chronos_t5_base_rounds01_12_inference_report.md"

if not DRIVE_CONTEXT_DIR.is_dir():
    raise FileNotFoundError(
        f"Context folder not found:\n  {DRIVE_CONTEXT_DIR}\n"
        "It must contain round_01_context.csv .. round_12_context.csv, copied "
        "from Data/contexts/.")

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Context folder: {DRIVE_CONTEXT_DIR}")
print(f"Output folder : {DRIVE_OUTPUT_DIR}")
print(f"Report will be: {REPORT_PATH.name}")

## 5. Check the 12 context files are present

In [ ]:
missing = [context_path(r).name for r in ROUNDS if not context_path(r).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing context files in {DRIVE_CONTEXT_DIR}:\n  " + "\n  ".join(missing)
        + "\n\nCopy round_01_context.csv .. round_12_context.csv there from "
          "Data/contexts/.")

print(f"All {len(list(ROUNDS))} context files present in {DRIVE_CONTEXT_DIR}")

## 6. GPU check and model loading

A GPU is required. The model is loaded once here and reused for all 12 rounds,
never reloaded inside the loop. Inference only: no training or fine-tuning.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Colab choose Runtime > Change runtime type > "
        "Hardware accelerator: GPU (T4 or better), then re-run this notebook "
        "from section 2. Refusing to run the 200M model on CPU."
    )

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

pipeline = ChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map="cuda",
    torch_dtype=DTYPE,
)

GPU_NAME = torch.cuda.get_device_name(0)
MODEL_LOADED_ONCE = True

print(f"GPU   : {GPU_NAME}")
print(f"dtype : {DTYPE}")
print(f"Loaded: {MODEL_ID} (inference only, no fine-tuning)")
print("Model loaded ONCE - it is reused for every round below.")

## 7. Helpers

Three steps per round. `load_and_validate_context` reads one context file and
checks its shape, dates, asset order and missing-value pattern.
`run_round_inference` draws the 100 trajectories. `save_and_verify_round` writes
the NPZ and reads it back to confirm it is intact.

In [ ]:
def load_and_validate_context(round_number: int):
    """Load one model context, validate it, and build the model input.

    Returns (context_matrix, context_df, forecast_dates). No preprocessing of
    any kind is applied to the returns.
    """
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    df = pd.read_csv(context_path(round_number), parse_dates=["date"])

    # --- Structural validation ---------------------------------------------
    assert df.shape[0] == CONTEXT_LENGTH, (
        f"Round {round_number}: expected {CONTEXT_LENGTH} rows, found {df.shape[0]}"
    )
    assert list(df.columns) == ["date"] + OFFICIAL_ASSET_ORDER, (
        f"Round {round_number}: columns are not 'date' + the official M6 asset order"
    )
    dates = df["date"]
    assert dates.is_monotonic_increasing, f"Round {round_number}: dates not ascending"
    assert not dates.duplicated().any(), f"Round {round_number}: duplicate dates"
    assert dates.iloc[0] == pd.Timestamp(ctx_start), (
        f"Round {round_number}: context starts {dates.iloc[0].date()}, expected {ctx_start}"
    )
    assert dates.iloc[-1] == pd.Timestamp(origin), (
        f"Round {round_number}: context ends {dates.iloc[-1].date()}, expected origin {origin}"
    )
    assert (dates <= pd.Timestamp(origin)).all(), (
        f"Round {round_number}: context contains a date after the origin"
    )

    # --- Model input: (100 assets, 512 time steps), values untouched --------
    matrix = df[OFFICIAL_ASSET_ORDER].to_numpy(dtype=np.float64).T
    assert matrix.shape == (N_ASSETS, CONTEXT_LENGTH)
    assert np.array_equal(
        matrix, df[OFFICIAL_ASSET_ORDER].to_numpy().T, equal_nan=True
    ), f"Round {round_number}: model input differs from the loaded context values"

    # --- Genuine leading missing history (must remain NaN) -----------------
    leading_missing, interior_missing = {}, {}
    for i, symbol in enumerate(OFFICIAL_ASSET_ORDER):
        row = matrix[i]
        n_lead = int(np.argmax(~np.isnan(row))) if np.isnan(row[0]) else 0
        if n_lead:
            leading_missing[symbol] = n_lead
        if np.isnan(row[n_lead:]).any():
            interior_missing[symbol] = int(np.isnan(row[n_lead:]).sum())
    assert not interior_missing, (
        f"Round {round_number}: unexpected interior NaNs {interior_missing}"
    )
    assert leading_missing == EXPECTED_LEADING_NAN[round_number], (
        f"Round {round_number}: leading NaN pattern changed - "
        f"expected {EXPECTED_LEADING_NAN[round_number]}, found {leading_missing}"
    )

    # --- Forecast dates -----------------------------------------------------
    forecast_dates = pd.bdate_range(fc_start, fc_end)
    assert len(forecast_dates) == PREDICTION_LENGTH, (
        f"Round {round_number}: forecast window is {len(forecast_dates)} weekdays, "
        f"expected {PREDICTION_LENGTH}"
    )

    return matrix, df, forecast_dates


def run_round_inference(context_matrix, round_number: int):
    """Forecast the 100 assets independently and return the raw (100,100,20) array."""
    # Reset the seed per round so any single round can be rerun reproducibly.
    torch.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)

    batch_outputs = []
    for start in range(0, N_ASSETS, SERIES_BATCH_SIZE):
        stop = min(start + SERIES_BATCH_SIZE, N_ASSETS)
        # One independent 1-D series per asset in this batch.
        batch_context = [
            torch.tensor(context_matrix[i], dtype=torch.float32)
            for i in range(start, stop)
        ]
        samples = pipeline.predict(
            inputs=batch_context,
            prediction_length=PREDICTION_LENGTH,
            num_samples=NUM_SAMPLES,
        )
        batch_outputs.append(samples.to(torch.float32).cpu().numpy())
        print(f"    assets {start + 1:3d}-{stop:3d} -> {tuple(batch_outputs[-1].shape)}")

    forecast_samples = np.concatenate(batch_outputs, axis=0)

    assert forecast_samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH), (
        f"Round {round_number}: expected "
        f"({N_ASSETS}, {NUM_SAMPLES}, {PREDICTION_LENGTH}), got {forecast_samples.shape}"
    )
    assert np.isfinite(forecast_samples).all(), (
        f"Round {round_number}: forecasts contain NaN or infinite values"
    )
    return forecast_samples


def save_and_verify_round(round_number: int, forecast_samples, forecast_dates):
    """Save the raw array to Drive, then reload it and verify it."""
    path = samples_path(round_number)
    date_strings = np.array([d.strftime("%Y-%m-%d") for d in forecast_dates])

    np.savez_compressed(
        path,
        forecast_samples=forecast_samples,
        asset_symbols=np.array(OFFICIAL_ASSET_ORDER),
        forecast_dates=date_strings,
    )

    with np.load(path, allow_pickle=False) as reloaded:
        r_samples = reloaded["forecast_samples"]
        r_symbols = reloaded["asset_symbols"]
        r_dates = reloaded["forecast_dates"]

    assert r_samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH), (
        f"Round {round_number}: reloaded shape {r_samples.shape}"
    )
    assert list(r_symbols) == OFFICIAL_ASSET_ORDER, (
        f"Round {round_number}: asset ordering changed on save/reload"
    )
    assert np.array_equal(r_samples, forecast_samples), (
        f"Round {round_number}: saved values differ from the forecasts"
    )
    assert np.isfinite(r_samples).all(), f"Round {round_number}: reloaded values not finite"
    assert list(r_dates) == list(date_strings), (
        f"Round {round_number}: forecast dates changed on save/reload"
    )
    return path


print("Helpers defined.")

### Run all 12 rounds

A failed round is recorded and the run continues, so one bad round does not
abandon the rest.

In [ ]:
round_records = []
failed_rounds = []
run_started_utc = datetime.now(timezone.utc)

for r in ROUNDS:
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[r]
    print(f"\n=== Round {r:02d} | origin {origin} | forecast {fc_start} .. {fc_end} ===")
    try:
        context_matrix, context_df, forecast_dates = load_and_validate_context(r)
        print(f"  context validated: {context_matrix.shape} (assets x time steps), "
              f"{context_df['date'].iloc[0].date()} .. {context_df['date'].iloc[-1].date()}")

        started = datetime.now(timezone.utc)
        forecast_samples = run_round_inference(context_matrix, r)
        elapsed = (datetime.now(timezone.utc) - started).total_seconds()

        all_finite = bool(np.isfinite(forecast_samples).all())
        saved_path = save_and_verify_round(r, forecast_samples, forecast_dates)

        round_records.append({
            "round": r,
            "context_start": str(context_df["date"].iloc[0].date()),
            "origin": str(context_df["date"].iloc[-1].date()),
            "forecast_start": str(forecast_dates[0].date()),
            "forecast_end": str(forecast_dates[-1].date()),
            "shape": tuple(forecast_samples.shape),
            "all_finite": all_finite,
            "npz_filename": saved_path.name,
            "seconds": round(elapsed, 1),
        })
        print(f"  raw output {tuple(forecast_samples.shape)} | all finite: {all_finite} "
              f"| {elapsed:.1f}s")
        print(f"  saved and verified -> {saved_path.name}")

    except Exception as exc:  # keep earlier rounds safe, report and continue
        failed_rounds.append((r, f"{type(exc).__name__}: {exc}"))
        print(f"  !! ROUND {r:02d} FAILED - earlier rounds remain saved")
        traceback.print_exc()

print(f"\nCompleted rounds: {[rec['round'] for rec in round_records]}")
if failed_rounds:
    print(f"FAILED rounds   : {[r for r, _ in failed_rounds]} (rerun these)")
else:
    print("No failures.")

## 8. Verify every saved output

An independent pass over what is actually on Drive: each NPZ is reopened and
checked for the expected arrays, shape, asset order, forecast dates and finite
values.

In [ ]:
verification_rows = []
ALL_ROUNDS_VERIFIED = True

for r in ROUNDS:
    path = samples_path(r)
    if not path.is_file():
        verification_rows.append((r, path.name, "MISSING", "-", "-", "-"))
        ALL_ROUNDS_VERIFIED = False
        continue

    _, _, fc_start, fc_end = ROUND_SCHEDULE[r]
    expected_dates = [d.strftime("%Y-%m-%d") for d in pd.bdate_range(fc_start, fc_end)]
    with np.load(path, allow_pickle=False) as f:
        keys = set(f.files)
        samples = f["forecast_samples"]
        symbols = list(f["asset_symbols"])
        fdates = list(f["forecast_dates"])

    ok_keys = keys == {"forecast_samples", "asset_symbols", "forecast_dates"}
    ok_shape = samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH)
    ok_order = symbols == OFFICIAL_ASSET_ORDER
    ok_dates = fdates == expected_dates
    ok_finite = bool(np.isfinite(samples).all())
    if not all([ok_keys, ok_shape, ok_order, ok_dates, ok_finite]):
        ALL_ROUNDS_VERIFIED = False

    verification_rows.append((
        r, path.name,
        "OK" if all([ok_keys, ok_shape, ok_order, ok_dates, ok_finite]) else "FAILED",
        str(samples.shape),
        "yes" if ok_order else "NO",
        "yes" if ok_finite else "NO",
    ))

print(f"{'Rnd':>3}  {'file':<44} {'status':<7} {'shape':<16} {'order':<6} finite")
for row in verification_rows:
    print(f"{row[0]:>3}  {row[1]:<44} {row[2]:<7} {row[3]:<16} {row[4]:<6} {row[5]}")

print(f"\nAll rounds 2-12 verified on Drive: {ALL_ROUNDS_VERIFIED}")

## 9. Inference report

A short provenance record written next to the outputs.

In [ ]:
if not round_records:
    raise RuntimeError("No rounds completed - nothing to report. Fix the failures above.")

table = "\n".join(
    f"| {rec['round']:02d} | {rec['origin']} | {rec['forecast_start']} .. {rec['forecast_end']} "
    f"| {rec['shape'][0]} x {rec['shape'][1]} x {rec['shape'][2]} "
    f"| {'all finite' if rec['all_finite'] else 'FAILED'} | `{rec['npz_filename']}` |"
    for rec in round_records
)

failure_note = (
    "- All 11 rounds (2-12) completed successfully.\n"
    if not failed_rounds
    else "- **Rounds that FAILED and must be rerun:** "
         + "; ".join(f"round {r} ({msg})" for r, msg in failed_rounds) + "\n"
)

report = f"""# Chronos-T5 Base (200M) - M6 Rounds 1-12 Inference Report

Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}
Run started: {run_started_utc.strftime('%Y-%m-%d %H:%M:%S UTC')}

## Experiment settings (identical for every round)

| Setting | Value |
|---|---|
| Model | `{MODEL_ID}` (Chronos-T5 Base, 200M parameters) |
| Context length | {CONTEXT_LENGTH} weekday daily log returns |
| Prediction length | {PREDICTION_LENGTH} weekdays |
| Sampled trajectories | {NUM_SAMPLES} per asset |
| Series batch size | {SERIES_BATCH_SIZE} (GPU efficiency only) |
| Random seed | {RANDOM_SEED} (reset at the start of every round) |
| Assets | {N_ASSETS} official M6 assets, official order |
| Runtime | Google Colab GPU ({GPU_NAME}), dtype {DTYPE} |
| Versions | chronos-forecasting {version('chronos-forecasting')}, torch {torch.__version__} |

- The model was **loaded once** before the round loop and reused for every round;
  it was never reloaded, trained or fine-tuned (`MODEL_LOADED_ONCE = {MODEL_LOADED_ONCE}`).
- Chronos is univariate: each of the {N_ASSETS} assets was forecast as its own
  independent series. Batching of {SERIES_BATCH_SIZE} assets per forward pass is a
  speed/memory measure only and cannot let one asset's history inform another's forecast.
- Contexts were taken unchanged from the context-building step files
  `round_XX_context.csv`, each verified byte-for-byte against the repository
  SHA-256 digests before inference. No preprocessing was repeated: no normalising,
  standardising, smoothing, clipping, averaging, outlier removal or NaN-filling.
  OGN's genuine leading missing values were left as NaN and handled by Chronos's
  missing-value mask.

## Per-round results

| Round | Origin (context end) | Forecast range | Output shape | Finite check | NPZ file |
|---|---|---|---|---|---|
{table}

{failure_note}
Each NPZ contains exactly `forecast_samples` (assets x sampled trajectories x
forecast weekdays), `asset_symbols` and `forecast_dates`. Every file was saved
immediately after its round finished, then reloaded and confirmed to open with the
correct shape, unchanged asset ordering, correct forecast dates and finite values.

## Context provenance (SHA-256)


## Scope and confirmations

- Raw sampled trajectories were saved exactly as produced by the model. **No
  post-processing** of any kind was applied: no averaging or aggregation of the
  trajectories, no four-week return conversion, no rescaling.
- **No quintile probabilities and no RPS evaluation** occurred in this notebook;
  no realised/ground-truth future returns were loaded.
- No asset ranking, no forecast-accuracy evaluation.
- **DRE:** raw Chronos output was preserved for every round, including the
  post-acquisition rounds (9-12) where non-zero values may appear. DRE forecasts
  were **not** overwritten or forced to zero here, and PLD was not used. The
  official M6 zero-return rule for DRE will be applied later, during the
  competition-specific post-processing/evaluation stage.
"""

REPORT_PATH.write_text(report, encoding="utf-8")
print(f"Report written to: {REPORT_PATH}")
print("Copy it into the repository as "
      "Results/Chronos_T5_Base_200M/reports/chronos_t5_base_rounds01_12_inference_report.md")

## 10. Completion summary

In [ ]:
print("Chronos-T5 Base (200M) - M6 Rounds 1-12")
print(f"  model             : {MODEL_ID} (loaded once, inference only)")
print(f"  rounds completed  : {len(round_records)} / {len(list(ROUNDS))} "
      f"-> {[rec['round'] for rec in round_records]}")
if failed_rounds:
    print(f"  rounds FAILED     : {[r for r, _ in failed_rounds]}")
print(f"  per-round shape   : ({N_ASSETS}, {NUM_SAMPLES}, {PREDICTION_LENGTH}) "
      "(assets x trajectories x forecast weekdays)")
print(f"  seed              : {RANDOM_SEED} (reset each round)")
print(f"  saved to          : {DRIVE_OUTPUT_DIR}")
print(f"  all files verified: {ALL_ROUNDS_VERIFIED}")
print(f"  report            : {REPORT_PATH.name}")
print("  post-processing   : none (no four-week returns, quintiles, or RPS)")
print("  DRE               : raw forecasts preserved, not zeroed (rule applied later)")
print("\nCopy back into the repository:")
print("  NPZ files -> Results/Chronos_T5_Base_200M/round_outputs/")
print("  report    -> Results/Chronos_T5_Base_200M/reports/")